## Data Profiling - CMS Medicare Inpatient Hospitals

Verifies the structure and quality of the raw CMS files before loading to Postgres. Checks grain uniqueness, type integrity, null patterns, suppression effects, and cross-year schema consistency.

**Source:** Medicare Inpatient Hospitals - by Provider and Service, FY2022-2024 (data.cms.gov)

**Outputs:** findings recorded in `docs/data_dictionary.md` and `docs/cleaning_decisions.md`

### Loading the csv files

In [1]:
import pandas as pd
import time, os

# loading key columns as strings
DTYPES_PRVSVC = {"Rndrng_Prvdr_CCN": str, "DRG_Cd": str, "Rndrng_Prvdr_Zip5": str}
DTYPES_PRV = {"Rndrng_Prvdr_CCN": str, "Rndrng_Prvdr_Zip5": str}
DTYPES_FID = {"Facility ID": str}

# files
FILES = {
    "prvsvc22": ("../data/raw/inpatient_provider_service/2022/MUP_INP_RY24_P03_V10_DY22_PrvSvc.csv", DTYPES_PRVSVC, "cp1252"),
    "prvsvc23": ("../data/raw/inpatient_provider_service/2023/MUP_INP_RY25_P03_V10_DY23_PrvSvc.csv", DTYPES_PRVSVC, "cp1252"),
    "prvsvc24": ("../data/raw/inpatient_provider_service/2024/MUP_INP_RY26_P03_V10_DY24_PrvSvc.csv", DTYPES_PRVSVC, "utf-8"),
    "prv22": ("../data/raw/inpatient_provider/2022/MUP_INP_RY24_P04_V10_DY22_Prv.csv", DTYPES_PRV, "cp1252"),
    "prv23": ("../data/raw/inpatient_provider/2023/MUP_INP_RY25_P04_V10_DY23_Prv.csv", DTYPES_PRV, "cp1252"),
    "prv24": ("../data/raw/inpatient_provider/2024/MUP_INP_RY26_P04_V10_DY24_Prv.csv", DTYPES_PRV, "utf-8"),
    "hgi": ("../data/raw/hospital_general_information.csv", DTYPES_FID, "utf-8"),
    "hrrp": ("../data/raw/hrrp_fy2026.csv", DTYPES_FID, "utf-8")

}

dfs = {}
stats = []

for name, (path, dtype, encoding) in FILES.items():
    size_mb = os.path.getsize(path) / 1024**2
    t0 = time.perf_counter()
    df = pd.read_csv(path, dtype=dtype, encoding=encoding, low_memory=False)
    elapsed = time.perf_counter() - t0

    dfs[name] = df
    stats.append({
        "file": name,
        "rows": len(df),
        "columns": df.shape[1],
        "size_mb": round(size_mb, 1),
        "mem_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 1),
        "load_s": round(elapsed, 2),
        "encoding": encoding
    })

load_summary = pd.DataFrame(stats)
load_summary

,file,rows,columns,size_mb,mem_mb,load_s,encoding
0,prvsvc22,145742,15,36.1,96.4,0.32,cp1252
1,prvsvc23,146427,15,36.4,96.9,0.32,cp1252
2,prvsvc24,145879,15,36.2,96.6,0.25,utf-8
3,prv22,3120,57,1.7,2.7,0.03,cp1252
4,prv23,3093,57,1.7,2.6,0.02,cp1252
5,prv24,3044,57,1.7,2.6,0.02,utf-8
6,hgi,5419,38,1.4,9.4,0.02,utf-8
7,hrrp,18330,12,2.0,8.0,0.02,utf-8


In [2]:
# verifying dtypes worked
KEY_COLS = {
    "prvsvc22": ["Rndrng_Prvdr_CCN", "DRG_Cd"],
    "prvsvc23": ["Rndrng_Prvdr_CCN", "DRG_Cd"],
    "prvsvc24": ["Rndrng_Prvdr_CCN", "DRG_Cd"],
    "prv22": ["Rndrng_Prvdr_CCN"],
    "prv23": ["Rndrng_Prvdr_CCN"],
    "prv24": ["Rndrng_Prvdr_CCN"],
    "hgi": ["Facility ID"],
    "hrrp": ["Facility ID"],
}

for name, cols in KEY_COLS.items():
    for col in cols:
        s = dfs[name][col]
        assert pd.api.types.is_string_dtype(s), f"{name}.{col} is {s.dtype}, expected string"
        print(f"{name:10} {col:20} {s.iloc[0]!r}")

prvsvc22   Rndrng_Prvdr_CCN     '010001'
prvsvc22   DRG_Cd               '023'
prvsvc23   Rndrng_Prvdr_CCN     '010001'
prvsvc23   DRG_Cd               '003'
prvsvc24   Rndrng_Prvdr_CCN     '010001'
prvsvc24   DRG_Cd               '003'
prv22      Rndrng_Prvdr_CCN     '010001'
prv23      Rndrng_Prvdr_CCN     '010001'
prv24      Rndrng_Prvdr_CCN     '010001'
hgi        Facility ID          '010001'
hrrp       Facility ID          '010001'


In [3]:
# important vars
PRVSVC = ["prvsvc22", "prvsvc23", "prvsvc24"]
PRV = ["prv22", "prv23","prv24"]
YEARS ={"prvsvc22": 2022, "prvsvc23": 2023, "prvsvc24": 2024}

CCN = "Rndrng_Prvdr_CCN"
DRG = "DRG_Cd"
DSCH = "Tot_Dschrgs"
CHRG = "Avg_Submtd_Cvrd_Chrg"
TPMT = "Avg_Tot_Pymt_Amt"
MPMT = "Avg_Mdcr_Pymt_Amt"

### Structure

In [4]:
# shapes and dtypes
for n in PRVSVC:
    print(f"\n{n}: {dfs[n].shape}")
    print(dfs[n].dtypes)

# cross-year column-name drift
base = set(dfs["prvsvc22"].columns)
for n in PRVSVC[1:]:
    cols = set(dfs[n].columns)
    print(f"\n{n} vs prvsvc22")
    print("\tonly in prvsvc22:", base - cols)
    print(f"\tonly in {n}:", cols - base)

# dtype drift on shared columns
for n in PRVSVC[1:]:
    for c in dfs["prvsvc22"].columns:
        if c in dfs[n].columns:
            t1, t2 = dfs["prvsvc22"][c].dtype, dfs[n][c].dtype
            if t1 != t2:
                print(f"{n}: {c} changed from {t1} to {t2}")

# grain test
for n in PRVSVC:
    dupes = dfs[n].duplicated(subset=[CCN, DRG]).sum()
    print(f"{n}: {dupes} duplicate (CCN, DRG) pairs")
    if dupes:
        display(dfs[n][dfs[n].duplicated(subset=[CCN, DRG], keep=False)].head(20))

# prv uniqueness
for n in PRV:
    print(f"{n}: {dfs[n].duplicated(subset=[CCN]).sum()} duplicate CCNs")


prvsvc22: (145742, 15)
Rndrng_Prvdr_CCN                 str
Rndrng_Prvdr_Org_Name            str
Rndrng_Prvdr_City                str
Rndrng_Prvdr_St                  str
Rndrng_Prvdr_State_FIPS        int64
Rndrng_Prvdr_Zip5                str
Rndrng_Prvdr_State_Abrvtn        str
Rndrng_Prvdr_RUCA            float64
Rndrng_Prvdr_RUCA_Desc           str
DRG_Cd                           str
DRG_Desc                         str
Tot_Dschrgs                    int64
Avg_Submtd_Cvrd_Chrg         float64
Avg_Tot_Pymt_Amt             float64
Avg_Mdcr_Pymt_Amt            float64
dtype: object

prvsvc23: (146427, 15)
Rndrng_Prvdr_CCN                 str
Rndrng_Prvdr_Org_Name            str
Rndrng_Prvdr_City                str
Rndrng_Prvdr_St                  str
Rndrng_Prvdr_State_FIPS        int64
Rndrng_Prvdr_Zip5                str
Rndrng_Prvdr_State_Abrvtn        str
Rndrng_Prvdr_RUCA            float64
Rndrng_Prvdr_RUCA_Desc           str
DRG_Cd                           str
DRG_Desc     

### Keys and Joins

In [5]:
SOURCES = {
    "prvsvc22": "Rndrng_Prvdr_CCN",
    "prvsvc23": "Rndrng_Prvdr_CCN",
    "prvsvc24": "Rndrng_Prvdr_CCN",
    "prv22": "Rndrng_Prvdr_CCN",
    "prv23": "Rndrng_Prvdr_CCN",
    "prv24": "Rndrng_Prvdr_CCN",
    "hgi": "Facility ID",
    "hrrp": "Facility ID",
}

# format and whitespace
for name, col in SOURCES.items():
    s = dfs[name][col]
    print(f"{name:10} lengths={dict(s.str.len().value_counts())} "
          f"whitespace={(s != s.str.strip()).sum()} distinct={s.nunique()}")

# join coverage against most recent claims year
claims = set(dfs["prvsvc24"][CCN])
for target in ["hgi", "hrrp"]:
    other = set(dfs[target]["Facility ID"])
    matched = claims & other
    print(f"\nprvsvc24 -> {target}: {len(matched)}/{len(claims)} "
          f"({len(matched)/len(claims):.1%}) matched")
    print(f"  unmatched claims CCNs: {len(claims - other)}")

# are unmatched hospitals systematically different?
hrrp_ccns = set(dfs["hrrp"]["Facility ID"])
vol = dfs["prvsvc24"].groupby(CCN)[DSCH].sum()
print("\nDischarges by HRRP match status:")
print(vol.groupby(vol.index.isin(hrrp_ccns)).describe())

# DRG codes
for n in PRVSVC:
    s = dfs[n][DRG]
    print(f"{n}: {s.nunique()} distinct DRGs, lengths={dict(s.str.len().value_counts())}")

prvsvc22   lengths={6: np.int64(145742)} whitespace=0 distinct=3015
prvsvc23   lengths={6: np.int64(146427)} whitespace=0 distinct=2945
prvsvc24   lengths={6: np.int64(145879)} whitespace=0 distinct=2906
prv22      lengths={6: np.int64(3120)} whitespace=0 distinct=3120
prv23      lengths={6: np.int64(3093)} whitespace=0 distinct=3093
prv24      lengths={6: np.int64(3044)} whitespace=0 distinct=3044
hgi        lengths={6: np.int64(5419)} whitespace=0 distinct=5419
hrrp       lengths={6: np.int64(18330)} whitespace=0 distinct=3055

prvsvc24 -> hgi: 2867/2906 (98.7%) matched
  unmatched claims CCNs: 39

prvsvc24 -> hrrp: 2883/2906 (99.2%) matched
  unmatched claims CCNs: 23

Discharges by HRRP match status:
        count         mean          std   min    25%    50%     75%      max
False    23.0    98.782609   120.197548  11.0   23.0   58.0   131.5    530.0
True   2883.0  1717.033992  2293.123619  11.0  257.5  883.0  2309.0  30767.0
prvsvc22: 533 distinct DRGs, lengths={3: np.int64(14574

In [6]:
# HGI attributes for dim_hospital
hgi = dfs["hgi"]
matched = hgi[hgi["Facility ID"].isin(set(dfs["prvsvc24"][CCN]))]
print(f"HGI rows matching 2024 claims: {len(matched)}")

for c in ["Hospital Type", "Hospital Ownership", "Emergency Services",
          "Hospital overall rating"]:
    print(f"\n{c}:")
    print(matched[c].value_counts(dropna=False))

HGI rows matching 2024 claims: 2867

Hospital Type:
Hospital Type
Acute Care Hospitals    2867
Name: count, dtype: int64

Hospital Ownership:
Hospital Ownership
Voluntary non-profit - Private                 1431
Proprietary                                     566
Voluntary non-profit - Other                    235
Voluntary non-profit - Church                   197
Government - Hospital District or Authority     196
Government - Local                              129
Physician                                        58
Government - State                               38
Government - Federal                             14
Tribal                                            3
Name: count, dtype: int64

Emergency Services:
Emergency Services
Yes    2644
No      223
Name: count, dtype: int64

Hospital overall rating:
Hospital overall rating
3                843
4                775
2                567
5                288
Not Available    226
1                168
Name: count, dtype: int64


### Types and Values

In [7]:
def sentinels(s):
    """Non-numeric strings hiding in a numeric-ish column."""
    return s[pd.to_numeric(s, errors="coerce").isna() & s.notna()].unique()

# PrvSvc: any object columns that should be numeric?
for n in PRVSVC:
    for c in [DSCH, CHRG, TPMT, MPMT]:
        if not pd.api.types.is_numeric_dtype(dfs[n][c]):
            print(f"{n}.{c} sentinels:", sentinels(dfs[n][c])[:20])

# zeros and negatives
for n in PRVSVC:
    for c in [DSCH, CHRG, TPMT, MPMT]:
        bad = (dfs[n][c] <= 0).sum()
        if bad:
            print(f"{n}.{c}: {bad} values <= 0")

# ordering check
for n in PRVSVC:
    d = dfs[n]
    v1 = (d[CHRG] < d[TPMT]).sum()
    v2 = (d[TPMT] < d[MPMT]).sum()
    print(f"{n}: charges<total={v1} ({v1/len(d):.2%}), total<medicare={v2} ({v2/len(d):.2%})")

# inspect a few violations if any
d = dfs["prvsvc24"]
display(d[d[CHRG] < d[TPMT]][[CCN, "Rndrng_Prvdr_Org_Name", DRG, "DRG_Desc", DSCH, CHRG, TPMT]].head(10))

# HRRP sentinels — find columns by partial match since names are long
hrrp = dfs["hrrp"]
print("\nHRRP columns:", hrrp.columns.tolist())
for c in hrrp.columns:
    if pd.api.types.is_string_dtype(hrrp[c]) and c not in ["Facility ID", "Facility Name", "State",
                                              "Measure Name", "Start Date", "End Date"]:
        print(f"  {c}: {sentinels(hrrp[c])[:10]}")

prvsvc22: charges<total=235 (0.16%), total<medicare=0 (0.00%)
prvsvc23: charges<total=281 (0.19%), total<medicare=0 (0.00%)
prvsvc24: charges<total=223 (0.15%), total<medicare=0 (0.00%)


,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,DRG_Cd,DRG_Desc,Tot_Dschrgs,Avg_Submtd_Cvrd_Chrg,Avg_Tot_Pymt_Amt
1222,010056,St Vincent's Birmingham,951,OTHER FACTORS INFLUENCING HEALTH STATUS,12,3838.583333,4498.000000
1224,010058,Bibb Medical Center,194,SIMPLE PNEUMONIA AND PLEURISY WITH CC,13,8925.615385,10466.846154
1225,010058,Bibb Medical Center,690,KIDNEY AND URINARY TRACT INFECTIONS WITHOUT MCC,12,7080.000000,10347.500000
1236,010062,Wiregrass Medical Center,895,"ALCOHOL, DRUG ABUSE OR DEPENDENCE WITH REHABIL...",23,12788.521739,13812.086957
1242,010073,Clay County Hospital,895,"ALCOHOL, DRUG ABUSE OR DEPENDENCE WITH REHABIL...",33,7896.939394,12693.484848
1609,010097,Elmore Community Hospital,895,"ALCOHOL, DRUG ABUSE OR DEPENDENCE WITH REHABIL...",28,5823.071429,12565.535714
4169,030071,Fort Defiance Indian Hospital,871,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOU...,20,14919.850000,21310.200000
4171,030073,Tuba City Regional Health Care Corporation,177,RESPIRATORY INFECTIONS AND INFLAMMATIONS WITH MCC,11,9159.818182,20590.909091
4172,030073,Tuba City Regional Health Care Corporation,189,PULMONARY EDEMA AND RESPIRATORY FAILURE,18,8929.611111,15950.777778
4173,030073,Tuba City Regional Health Care Corporation,193,SIMPLE PNEUMONIA AND PLEURISY WITH MCC,12,9396.083333,16847.583333



HRRP columns: ['Facility Name', 'Facility ID', 'State', 'Measure Name', 'Number of Discharges', 'Footnote', 'Excess Readmission Ratio', 'Predicted Readmission Rate', 'Expected Readmission Rate', 'Number of Readmissions', 'Start Date', 'End Date']
  Number of Readmissions: <StringArray>
['Too Few to Report']
Length: 1, dtype: str


### Nulls and Suppression

In [8]:
# nulls
for n in PRVSVC:
    nulls = dfs[n].isna().sum()
    print(f"\n{n}:\n{nulls[nulls > 0]}" if nulls.any() else f"{n}: no nulls")

# discharge distribution
for n in PRVSVC:
    print(f"\n{n} {DSCH}:")
    print(dfs[n][DSCH].describe())
    print(dfs[n][DSCH].quantile([.1,.25,.5,.75,.9,.99]))

# verifying suppression threshold against the methodology PDF
for n in PRVSVC:
    print(f"{n}: min discharges = {dfs[n][DSCH].min()}")

# threshold loss table
rows = []
for n in PRVSVC:
    d = dfs[n]
    tot_rows, tot_dsch = len(d), d[DSCH].sum()
    for t in [11, 25, 50]:
        keep = d[d[DSCH] >= t]
        rows.append({
            "file": n, "threshold": t,
            "rows_kept_pct": round(len(keep)/tot_rows*100, 1),
            "dschg_kept_pct": round(keep[DSCH].sum()/tot_dsch*100, 1),
        })
pd.DataFrame(rows)


prvsvc22:
Rndrng_Prvdr_RUCA         677
Rndrng_Prvdr_RUCA_Desc    677
dtype: int64

prvsvc23:
Rndrng_Prvdr_RUCA         4
Rndrng_Prvdr_RUCA_Desc    4
dtype: int64
prvsvc24: no nulls

prvsvc22 Tot_Dschrgs:
count    145742.000000
mean         34.369653
std          50.314022
min          11.000000
25%          14.000000
50%          20.000000
75%          35.000000
max        2750.000000
Name: Tot_Dschrgs, dtype: float64
0.10     12.00
0.25     14.00
0.50     20.00
0.75     35.00
0.90     64.00
0.99    246.59
Name: Tot_Dschrgs, dtype: float64

prvsvc23 Tot_Dschrgs:
count    146427.000000
mean         33.875754
std          49.814354
min          11.000000
25%          14.000000
50%          20.000000
75%          35.000000
max        3210.000000
Name: Tot_Dschrgs, dtype: float64
0.10     12.0
0.25     14.0
0.50     20.0
0.75     35.0
0.90     63.0
0.99    239.0
Name: Tot_Dschrgs, dtype: float64

prvsvc24 Tot_Dschrgs:
count    145879.000000
mean         33.949239
std          50.571682
m

,file,threshold,rows_kept_pct,dschg_kept_pct
0,prvsvc22,11,100.0,100.0
1,prvsvc22,25,40.1,72.5
2,prvsvc22,50,15.1,47.8
3,prvsvc23,11,100.0,100.0
4,prvsvc23,25,39.7,71.8
5,prvsvc23,50,14.8,46.8
6,prvsvc24,11,100.0,100.0
7,prvsvc24,25,40.0,72.0
8,prvsvc24,50,14.9,46.9


In [9]:
# HRRP suppression
err_col = [c for c in hrrp.columns if "Excess" in c][0]
hrrp["_err"] = pd.to_numeric(hrrp[err_col], errors="coerce")

summary = hrrp.groupby("Measure Name")["_err"].agg(total="size", usable="count")
summary["pct"] = (summary["usable"] / summary["total"] * 100).round(1)
print(summary)

per_hosp = hrrp.groupby("Facility ID")["_err"].count()
print("\nMeasures available per hospital:")
print(per_hosp.value_counts().sort_index())

                        total  usable   pct
Measure Name                               
READM-30-AMI-HRRP        3055    1736  56.8
READM-30-CABG-HRRP       3055     878  28.7
READM-30-COPD-HRRP       3055    2323  76.0
READM-30-HF-HRRP         3055    2621  85.8
READM-30-HIP-KNEE-HRRP   3055    1447  47.4
READM-30-PN-HRRP         3055    2715  88.9

Measures available per hospital:
_err
0    222
1    202
2    239
3    484
4    599
5    662
6    647
Name: count, dtype: int64


### Distribution

In [10]:
d = dfs["prvsvc24"]

# DRGs and discharges per hospital
per_hosp = d.groupby(CCN).agg(drgs=(DRG, "nunique"), dschgs=(DSCH, "sum"))
print(per_hosp.describe())

# extremes
display(d.nlargest(10, TPMT)[[CCN, "Rndrng_Prvdr_Org_Name", DRG, "DRG_Desc", DSCH, TPMT]])
display(d.nsmallest(10, TPMT)[[CCN, "Rndrng_Prvdr_Org_Name", DRG, "DRG_Desc", DSCH, TPMT]])

# payment spread within single DRGs
top_drgs = d.groupby([DRG, "DRG_Desc"])[DSCH].sum().nlargest(5)
print(top_drgs)

for (code, desc) in top_drgs.index:
    sub = d[d[DRG] == code][TPMT]
    p10, p50, p90 = sub.quantile([.1, .5, .9])
    print(f"\nDRG {code} — {desc[:55]}")
    print(f"  n={len(sub)} hospitals  p10=${p10:,.0f}  median=${p50:,.0f}  "
          f"p90=${p90:,.0f}  ratio={p90/p10:.2f}x")

              drgs        dschgs
count  2906.000000   2906.000000
mean     50.199243   1704.226084
std      53.114385   2288.550174
min       1.000000     11.000000
25%      10.000000    251.250000
50%      32.000000    867.500000
75%      71.000000   2281.750000
max     387.000000  30767.000000


,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,DRG_Cd,DRG_Desc,Tot_Dschrgs,Avg_Tot_Pymt_Amt
50352,150084,Ascension St Vincent Hospital,466,REVISION OF HIP OR KNEE REPLACEMENT WITH MCC,18,1.443310e+06
47468,140276,Loyola University Medical Center,018,CHIMERIC ANTIGEN RECEPTOR (CAR) T-CELL AND OTH...,11,9.505475e+05
71629,240010,Mayo Clinic Hospital Rochester,018,CHIMERIC ANTIGEN RECEPTOR (CAR) T-CELL AND OTH...,19,6.571616e+05
110660,390027,Temple University Hospital,018,CHIMERIC ANTIGEN RECEPTOR (CAR) T-CELL AND OTH...,16,6.399649e+05
24401,090004,Medstar Georgetown University Hospital,018,CHIMERIC ANTIGEN RECEPTOR (CAR) T-CELL AND OTH...,15,6.386277e+05
16309,050441,Stanford Health Care,003,ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRIN...,64,6.241693e+05
13178,050262,Ronald Reagan Ucla Medical Center,001,HEART TRANSPLANT OR IMPLANT OF HEART ASSIST SY...,25,6.119297e+05
8623,050047,California Pacific Medical Center- Van Ness Ca...,001,HEART TRANSPLANT OR IMPLANT OF HEART ASSIST SY...,19,5.951264e+05
99052,340113,Carolinas Medical Center/Behav Health,018,CHIMERIC ANTIGEN RECEPTOR (CAR) T-CELL AND OTH...,27,5.947555e+05
47662,140281,Northwestern Memorial Hospital,018,CHIMERIC ANTIGEN RECEPTOR (CAR) T-CELL AND OTH...,36,5.869857e+05


,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,DRG_Cd,DRG_Desc,Tot_Dschrgs,Avg_Tot_Pymt_Amt
60452,210017,Garrett Regional Medical Center,951,OTHER FACTORS INFLUENCING HEALTH STATUS,13,1849.076923
133953,470001,Central Vermont Medical Center,310,CARDIAC ARRHYTHMIA AND CONDUCTION DISORDERS WI...,15,2102.866667
133952,470001,Central Vermont Medical Center,309,CARDIAC ARRHYTHMIA AND CONDUCTION DISORDERS WI...,18,2408.055556
133957,470001,Central Vermont Medical Center,389,GASTROINTESTINAL OBSTRUCTION WITH CC,15,2502.466667
134145,470005,Rutland Regional Medical Center,282,"ACUTE MYOCARDIAL INFARCTION, DISCHARGED ALIVE ...",19,3095.315789
133968,470001,Central Vermont Medical Center,641,"MISCELLANEOUS DISORDERS OF NUTRITION, METABOLI...",21,3202.190476
133959,470001,Central Vermont Medical Center,392,"ESOPHAGITIS, GASTROENTERITIS AND MISCELLANEOUS...",13,3257.153846
7217,040134,"Arkansas Heart Hospital, Llc",310,CARDIAC ARRHYTHMIA AND CONDUCTION DISORDERS WI...,173,3449.543353
105163,360155,Southwest General Health Center,310,CARDIAC ARRHYTHMIA AND CONDUCTION DISORDERS WI...,11,3488.818182
121533,440029,Williamson Medical Center,310,CARDIAC ARRHYTHMIA AND CONDUCTION DISORDERS WI...,11,3531.363636


DRG_Cd  DRG_Desc                                                    
871     SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS WITH MCC       577119
291     HEART FAILURE AND SHOCK WITH MCC                                304694
177     RESPIRATORY INFECTIONS AND INFLAMMATIONS WITH MCC               141351
193     SIMPLE PNEUMONIA AND PLEURISY WITH MCC                          136649
872     SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS WITHOUT MCC    103998
Name: Tot_Dschrgs, dtype: int64

DRG 871 — SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS WITH M
  n=2661 hospitals  p10=$13,355  median=$16,182  p90=$23,576  ratio=1.77x

DRG 291 — HEART FAILURE AND SHOCK WITH MCC
  n=2587 hospitals  p10=$8,933  median=$10,959  p90=$15,748  ratio=1.76x

DRG 177 — RESPIRATORY INFECTIONS AND INFLAMMATIONS WITH MCC
  n=2332 hospitals  p10=$11,348  median=$13,722  p90=$19,256  ratio=1.70x

DRG 193 — SIMPLE PNEUMONIA AND PLEURISY WITH MCC
  n=2442 hospitals  p10=$9,269  median=$11,253  p90=$15,494  

### Cross-year

In [11]:
ccn_sets = {n: set(dfs[n][CCN]) for n in PRVSVC}
drg_sets = {n: set(dfs[n][DRG]) for n in PRVSVC}

all_three = set.intersection(*ccn_sets.values())
print(f"CCNs in all 3 years: {len(all_three)}")
for n in PRVSVC:
    print(f"  {n}: {len(ccn_sets[n])} total, {len(ccn_sets[n] - all_three)} not in all 3")

print(f"\nDRGs in all 3 years: {len(set.intersection(*drg_sets.values()))}")
print("  entered by 2024:", sorted(drg_sets['prvsvc24'] - drg_sets['prvsvc22'])[:20])
print("  gone since 2022:", sorted(drg_sets['prvsvc22'] - drg_sets['prvsvc24'])[:20])

for n in PRVSVC:
    print(f"{YEARS[n]}: {dfs[n][DSCH].sum():,} discharges, {dfs[n][CCN].nunique()} hospitals")

CCNs in all 3 years: 2850
  prvsvc22: 3015 total, 165 not in all 3
  prvsvc23: 2945 total, 95 not in all 3
  prvsvc24: 2906 total, 56 not in all 3

DRGs in all 3 years: 488
  entered by 2024: ['021', '090', '097', '113', '124', '147', '173', '212', '231', '275', '276', '277', '278', '279', '317', '321', '322', '323', '324', '347']
  gone since 2022: ['008', '052', '059', '063', '098', '114', '143', '218', '222', '224', '225', '226', '227', '232', '246', '247', '248', '249', '339', '342']
2022: 5,009,102 discharges, 3015 hospitals
2023: 4,960,325 discharges, 2945 hospitals
2024: 4,952,481 discharges, 2906 hospitals


### Validate against Provider

In [12]:
# check against prv22.columns first
print([c for c in dfs["prv22"].columns if "Dschrg" in c or "Pymt" in c])

['Tot_Pymt_Amt', 'Tot_Mdcr_Pymt_Amt', 'Tot_Dschrgs']


In [13]:
svc = dfs["prvsvc22"].copy()
prv = dfs["prv22"]

PRV_DSCH = "Tot_Dschrgs"
PRV_TPMT = "Tot_Pymt_Amt"

# weighted totals: the Avg_ columns are per-discharge averages, not totals
svc["total_pymt"] = svc[DSCH] * svc[TPMT]

agg = svc.groupby(CCN).agg(
    svc_dschgs=(DSCH, "sum"),
    svc_total_pymt=("total_pymt", "sum"),
)
agg["svc_avg_pymt"] = agg["svc_total_pymt"] / agg["svc_dschgs"]

prv_cols = prv.set_index(CCN)[[PRV_DSCH, PRV_TPMT]].rename(
    columns={PRV_DSCH: "prv_dschgs", PRV_TPMT: "prv_total_pymt"}
)

cmp = agg.join(prv_cols, how="inner")
cmp["prv_avg_pymt"] = cmp["prv_total_pymt"] / cmp["prv_dschgs"]
cmp["coverage"] = cmp["svc_dschgs"] / cmp["prv_dschgs"]
cmp["pymt_ratio"] = cmp["svc_avg_pymt"] / cmp["prv_avg_pymt"]

# discharge coverage
print(f"Hospitals compared: {len(cmp)}")
print("\nDischarge coverage per hospital:")
print(cmp["coverage"].describe())
print(f"\nHospitals with coverage above 100%: {(cmp['coverage'] > 1.0).sum()}")
print(f"Overall discharge coverage: {cmp['svc_dschgs'].sum() / cmp['prv_dschgs'].sum():.2%}")

# payment comparison
print("\nPayment ratio per hospital (service-level / provider-level):")
print(cmp["pymt_ratio"].describe())

svc_national = cmp["svc_total_pymt"].sum() / cmp["svc_dschgs"].sum()
prv_national = cmp["prv_total_pymt"].sum() / cmp["prv_dschgs"].sum()
print(f"\nService-level national avg:  ${svc_national:,.2f}")
print(f"Provider-level national avg: ${prv_national:,.2f}")
print(f"Ratio: {svc_national / prv_national:.3f}")

display(cmp[cmp["coverage"] > 1.0].head(10))    # should be empty

Hospitals compared: 3015

Discharge coverage per hospital:
count    3015.000000
mean        0.582527
std         0.181070
min         0.053942
25%         0.464393
50%         0.603204
75%         0.716636
max         1.000000
Name: coverage, dtype: float64

Hospitals with coverage above 100%: 0
Overall discharge coverage: 71.81%

Payment ratio per hospital (service-level / provider-level):
count    3015.000000
mean        0.989663
std         0.129388
min         0.471980
25%         0.929965
50%         0.965295
75%         1.013039
max         3.735005
Name: pymt_ratio, dtype: float64

Service-level national avg:  $17,772.24
Provider-level national avg: $18,004.54
Ratio: 0.987


,svc_dschgs,svc_total_pymt,svc_avg_pymt,prv_dschgs,prv_total_pymt,prv_avg_pymt,coverage,pymt_ratio
Rndrng_Prvdr_CCN,,,,,,,,


In [14]:
# does low coverage explain the extreme payment ratios?
print(cmp[["coverage", "pymt_ratio"]].corr())

bands = pd.cut(cmp["coverage"], [0, .25, .5, .75, 1.0])
print("\nPayment ratio by coverage band:")
print(cmp.groupby(bands, observed=True)["pymt_ratio"].describe())

print("\nHospitals below 25% coverage:", (cmp["coverage"] < 0.25).sum())

            coverage  pymt_ratio
coverage    1.000000   -0.364796
pymt_ratio -0.364796    1.000000

Payment ratio by coverage band:
              count      mean       std       min       25%       50%  \
coverage                                                                
(0.0, 0.25]   158.0  1.210366  0.371352  0.548788  1.004661  1.166963   
(0.25, 0.5]   756.0  1.026172  0.128372  0.471980  0.938910  1.015987   
(0.5, 0.75]  1531.0  0.954769  0.057112  0.760750  0.919282  0.947845   
(0.75, 1.0]   570.0  0.973788  0.034139  0.850870  0.949807  0.970433   

                  75%       max  
coverage                         
(0.0, 0.25]  1.336991  3.735005  
(0.25, 0.5]  1.096993  1.646458  
(0.5, 0.75]  0.981260  1.309963  
(0.75, 1.0]  0.996643  1.116346  

Hospitals below 25% coverage: 157


### Anchor Number

In [15]:
anchors = {}
for n in PRVSVC:
    d = dfs[n]
    anchors[YEARS[n]] = (d[DSCH] * d[TPMT]).sum() / d[DSCH].sum()

for yr, v in anchors.items():
    print(f"{yr}: ${v:,.2f} weighted avg payment per discharge")

# The wrong way, for contrast — this is the error you're guarding against
for n in PRVSVC:
    print(f"{YEARS[n]} naive mean: ${dfs[n][TPMT].mean():,.2f}")

2022: $17,772.24 weighted avg payment per discharge
2023: $17,823.65 weighted avg payment per discharge
2024: $18,359.99 weighted avg payment per discharge
2022 naive mean: $18,505.73
2023 naive mean: $18,512.96
2024 naive mean: $19,151.32
